In [1]:
import pandas as pd
import ftfy
import random
from collections import defaultdict
from datasets import Dataset, DatasetDict
from preprocess import transcriptions

c:\Users\auran\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\auran\OneDrive\Documents\ensae\3A\NLP\Projet-de-NLP\preprocess.py:18: DtypeWarning: Columns (8,9,10,12,28,29,30,31,32,33,34,35,36,37,38,39,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("data/metadonnees.csv")


In [2]:
# Chargement des données et gestion des problèmes d'encodage
annotations = pd.read_csv("annotation_token_strategy_annotations.csv", sep=";",encoding="latin-1")
annotations = annotations.apply(lambda col: col.map(lambda x: ftfy.fix_text(x) if isinstance(x, str) else x))
annotations = annotations.dropna()
# Erreurs de frappe constatée a posteriori
annotations["label_concret"] = annotations["label_concret"].replace('B-VERB_CONRET', 'B-VERB_CONCRET')
annotations["label_concret"] = annotations["label_concret"].replace('B-VERB_CONCRET ', 'B-VERB_CONCRET')
annotations["label_concret"] = annotations["label_concret"].replace('I-VERB-CONCRET', 'I-VERB_CONCRET')
annotations["label_concret"] = annotations["label_concret"].replace('E-VERB-CONCRET', 'E-VERB_CONCRET')
annotations["label_concret"] = annotations["label_concret"].replace('B-VERB-CONTEXTE', 'B-VERB_CONTEXTE')  
textes_annotes = pd.unique(annotations["doc_id_origine"])

annotations_work_concret = annotations[["phrase_id", "token_text", "label_concret"]]
annotations_work_concret = annotations_work_concret.groupby("phrase_id").agg(
    token_text=("token_text", list),
    label_concret=("label_concret", lambda x: [v[0] if isinstance(v, list) else v for v in x])
).reset_index()


In [3]:
# Préparation pour avoir une partie train et une partie test

def split_by_phrase_id(df, col_tag, train_ratio=0.8, val_ratio=0.1, seed=42):
    
    sentences = df.to_dict(orient="records")

    random.seed(seed)
    random.shuffle(sentences)

    n = len(sentences)

    def density_bucket(example):
        n_entities = sum(1 for t in example[col_tag] if t.startswith("B-"))
        if n_entities == 0:
            return 0
        elif n_entities <= 2:
            return 1
        else:
            return 2

    buckets = {0: [], 1: [], 2: []}
    for _, ex in df.iterrows():
        buckets[density_bucket(ex)].append(ex)

    train, val, test = [], [], []

    for bucket_data in buckets.values():
        random.shuffle(bucket_data)
        n = len(bucket_data)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)

        train += bucket_data[:n_train]
        val   += bucket_data[n_train:n_train + n_val]
        test  += bucket_data[n_train + n_val:]

    random.shuffle(train)
    random.shuffle(val)
    random.shuffle(test)

    print(f"Train : {len(train)} phrases")
    print(f"Val   : {len(val)} phrases")
    print(f"Test  : {len(test)} phrases")

    return train, val, test

train, val, test = split_by_phrase_id(annotations_work_concret, "label_concret")

Train : 170 phrases
Val   : 21 phrases
Test  : 23 phrases


In [4]:
labels_conc = list({t for row in annotations_work_concret["label_concret"] for t in row})
num_labels_conc = len(labels_conc)
id2label = {id:label for id, label in enumerate(labels_conc)}
label2id = {label:id  for id, label in enumerate(labels_conc)}

def encode_labels(example, col_tag):
    example[col_tag] = [label2id[t] for t in example[col_tag]]
    return example

COL_TAG = "label_concret"
train = [row.to_dict() for row in train]
test  = [row.to_dict() for row in test]
val   = [row.to_dict() for row in val]

train_processed = [encode_labels(ex, COL_TAG) for ex in train]
test_processed = [encode_labels(ex, COL_TAG) for ex in test]
val_processed = [encode_labels(ex, COL_TAG) for ex in val]

train_dataset = Dataset.from_list(train_processed)
test_dataset = Dataset.from_list(test_processed)
val_dataset = Dataset.from_list(val_processed)

dataset = DatasetDict({"train": train_dataset, "test":test_dataset, "validation": val_dataset})

In [5]:
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification

MODEL_NAME = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["token_text"],
        col_tag = COL_TAG,
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,
    )

    all_labels = []
    for i, labels in enumerate(examples[COL_TAG]):
        word_ids = tokenized.word_ids(batch_index=i)
        print(f"Nb labels: {len(labels)}, max word_id: {max(w for w in word_ids if w is not None)}")
        aligned_labels = []
        prev_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != prev_word_id:
                aligned_labels.append(labels[word_id])
            else:
                aligned_labels.append(-100)
            prev_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

Map: 100%|██████████| 170/170 [00:00<00:00, 2479.95 examples/s]


Nb labels: 36, max word_id: 35
Nb labels: 77, max word_id: 76
Nb labels: 18, max word_id: 17
Nb labels: 10, max word_id: 9
Nb labels: 28, max word_id: 27
Nb labels: 39, max word_id: 38
Nb labels: 31, max word_id: 30
Nb labels: 29, max word_id: 28
Nb labels: 31, max word_id: 30
Nb labels: 36, max word_id: 35
Nb labels: 97, max word_id: 96
Nb labels: 40, max word_id: 39
Nb labels: 35, max word_id: 34
Nb labels: 31, max word_id: 30
Nb labels: 56, max word_id: 55
Nb labels: 43, max word_id: 42
Nb labels: 26, max word_id: 25
Nb labels: 110, max word_id: 96
Nb labels: 47, max word_id: 46
Nb labels: 49, max word_id: 48
Nb labels: 54, max word_id: 53
Nb labels: 74, max word_id: 73
Nb labels: 38, max word_id: 37
Nb labels: 24, max word_id: 23
Nb labels: 60, max word_id: 59
Nb labels: 33, max word_id: 32
Nb labels: 17, max word_id: 16
Nb labels: 17, max word_id: 16
Nb labels: 19, max word_id: 18
Nb labels: 26, max word_id: 25
Nb labels: 21, max word_id: 20
Nb labels: 29, max word_id: 28
Nb label

Map: 100%|██████████| 23/23 [00:00<00:00, 3026.00 examples/s]

Nb labels: 16, max word_id: 15
Nb labels: 45, max word_id: 44
Nb labels: 26, max word_id: 25
Nb labels: 57, max word_id: 56
Nb labels: 34, max word_id: 33
Nb labels: 43, max word_id: 42
Nb labels: 56, max word_id: 55
Nb labels: 9, max word_id: 8
Nb labels: 21, max word_id: 20
Nb labels: 27, max word_id: 26
Nb labels: 41, max word_id: 40
Nb labels: 13, max word_id: 12
Nb labels: 46, max word_id: 45
Nb labels: 41, max word_id: 40
Nb labels: 59, max word_id: 58
Nb labels: 55, max word_id: 54
Nb labels: 101, max word_id: 100
Nb labels: 18, max word_id: 17
Nb labels: 44, max word_id: 43
Nb labels: 18, max word_id: 17
Nb labels: 28, max word_id: 27
Nb labels: 16, max word_id: 15
Nb labels: 63, max word_id: 62



Map: 100%|██████████| 21/21 [00:00<00:00, 3134.31 examples/s]

Nb labels: 37, max word_id: 36
Nb labels: 25, max word_id: 24
Nb labels: 38, max word_id: 37
Nb labels: 18, max word_id: 17
Nb labels: 28, max word_id: 27
Nb labels: 23, max word_id: 22
Nb labels: 42, max word_id: 41
Nb labels: 35, max word_id: 34
Nb labels: 113, max word_id: 83
Nb labels: 13, max word_id: 12
Nb labels: 72, max word_id: 71
Nb labels: 79, max word_id: 78
Nb labels: 62, max word_id: 61
Nb labels: 29, max word_id: 28
Nb labels: 28, max word_id: 27
Nb labels: 12, max word_id: 11
Nb labels: 51, max word_id: 50
Nb labels: 39, max word_id: 38
Nb labels: 8, max word_id: 7
Nb labels: 49, max word_id: 48
Nb labels: 57, max word_id: 56


In [23]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import numpy as np
import evaluate

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels_conc),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4080.57it/s]
CamembertForTokenClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
import torch
from collections import Counter
# Métrique seqeval (F1 par entité)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        p_row, l_row = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100: 
                p_row.append(id2label[p])
                l_row.append(id2label[l])
        true_preds.append(p_row)
        true_labels.append(l_row)

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

all_labels_flat = [t for ex in train for t in ex["label_concret"]]
counts = Counter(all_labels_flat)
total  = sum(counts.values())
weights = torch.zeros(num_labels_conc)
for label_id, count in counts.items():
    weights[label_id] = total / (num_labels_conc * count)
print("Poids par classe :", {id2label[i]: round(w.item(), 2) for i, w in enumerate(weights)})

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=weights.to(logits.device),
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, num_labels_conc), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Collator : gère le padding dynamique
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./camembert-ner",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-5,
    weight_decay=0.00,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=1,
    fp16=False,             # Activez si GPU compatible
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Sauvegarde du meilleur modèle
trainer.save_model("./camembert-ner-final")
tokenizer.save_pretrained("./camembert-ner-final")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Poids par classe : {'I-VERB_CONCRET': 4.97, 'I-VERB_CONTEXTE': 43.19, 'B-OBJ': 26.58, 'B-SUJ': 3.13, 'O': 0.12, 'B-VERB_CONTEXTE': 5.12, 'E-VERB_CONCRET': 15.7, 'B-VERB_CONCRET': 7.12, 'E-VERB_CONTEXTE': 172.75}


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,2.118039,2.071172,0.050734,0.622951,0.093827,0.126659
2,1.978253,1.963750,0.063636,0.688525,0.116505,0.234017
3,2.011146,1.893362,0.077739,0.721311,0.140351,0.357057
4,1.867124,1.847368,0.093750,0.737705,0.166359,0.464415
5,1.813538,1.832593,0.100437,0.754098,0.177264,0.483715


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]
c:\Users\auran\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]
c:\Users\auran\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.24s/it]
c:\Users\auran\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|████

('./camembert-ner-final\\tokenizer_config.json',
 './camembert-ner-final\\tokenizer.json')

In [25]:
# Courbes d'apprentissage du modèle
import json
import plotly.express as px 

with open("./camembert-ner/checkpoint-215/trainer_state.json", "r") as file: 
    training_state = json.load(file)

loss = []

for log in training_state ["log_history"]:
    step = log["step"]
    if "loss" in log:
        loss += [{"step": step, "loss": log["loss"], "split": "train"}]
    elif "eval_loss" in log:
        loss += [{"step": step, "loss": log["eval_loss"], "split": "eval"}]
    else: 
        # thweird
        print(log)

loss = pd.DataFrame(loss)

px.line(loss, x = "step", y = "loss", color =  "split") 

In [26]:
labels_true = []
labels_pred = []

model.eval()

for batch in tokenized_dataset["test"].batch(batch_size=4, drop_last_batch=False):
    model_input = {
        "input_ids": torch.tensor(batch["input_ids"]),
        "attention_mask": torch.tensor(batch["attention_mask"]),
    }

    with torch.no_grad():
        logits = model(**model_input).logits  # (batch, seq_len, num_labels)

    predictions = torch.argmax(logits, dim=-1).numpy()  # (batch, seq_len)
    gold_labels = np.array(batch["labels"])              # (batch, seq_len)

    for pred_seq, label_seq in zip(predictions, gold_labels):
        for p, l in zip(pred_seq, label_seq):
            if l != -100:  # ✅ ignorer les tokens spéciaux et le padding
                labels_pred.append(id2label[int(p)])
                labels_true.append(id2label[int(l)])

(
    pd.DataFrame({
        "predict": labels_pred,
        "gold_standard": labels_true,
    })
    .to_csv("./outputs/predictions_ner.csv", index=False)
)

Batching examples: 100%|██████████| 23/23 [00:00<00:00, 1296.85 examples/s]


In [27]:
from sklearn.metrics import classification_report

print(classification_report(y_true = labels_true, y_pred = labels_pred))

                 precision    recall  f1-score   support

          B-OBJ       0.27      1.00      0.43         6
          B-SUJ       0.13      1.00      0.23        27
 B-VERB_CONCRET       0.07      0.33      0.12         9
B-VERB_CONTEXTE       0.16      0.84      0.27        19
 E-VERB_CONCRET       0.01      1.00      0.03         1
 I-VERB_CONCRET       0.03      0.25      0.05         4
I-VERB_CONTEXTE       0.00      0.00      0.00         3
              O       1.00      0.50      0.66       808

       accuracy                           0.52       877
      macro avg       0.21      0.62      0.22       877
   weighted avg       0.93      0.52      0.63       877



c:\Users\auran\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\auran\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\auran\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave